# Notebook 01 - U²-Netp Demo

Load model, forward pass, train 1 epoch dummy, visualize output.

In [ ]:
import os, sys
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')
sys.path.insert(0, '..')
import torch
from ml2.u2net.model import U2NETp
from ml2.u2net.loss import ComboLoss

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
model = U2NETp().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Params: {n_params:,} (~{n_params * 4 / 1e6:.1f}MB fp32)')

x = torch.randn(2, 3, 320, 320, device=device)
outs = model(x)
print(f'Outputs: {len(outs)} tensors, fused: {outs[0].shape}')

In [ ]:
# Forward + loss
criterion = ComboLoss()
tgt = torch.randint(0, 2, (2, 1, 320, 320), device=device).float()
loss, per_side = criterion(outs, tgt)
print(f'Loss: {loss.item():.4f}')
print(f'Per-side: {per_side}')

In [ ]:
# Train 1 step
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
model.train()
optimizer.zero_grad()
outs = model(x)
loss, _ = criterion(outs, tgt)
loss.backward()
optimizer.step()
print(f'Step done. Loss: {loss.item():.4f}')

In [ ]:
# Visualize fused output
import matplotlib.pyplot as plt
model.eval()
with torch.no_grad():
    out = torch.sigmoid(model(x)[0])
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(x[0].permute(1, 2, 0).cpu(), cmap='gray')
axes[0].set_title('Input'); axes[0].axis('off')
axes[1].imshow(out[0, 0].cpu(), cmap='gray')
axes[1].set_title('Prediction'); axes[1].axis('off')
plt.show()